In [1]:
!pip install -U spacy[cuda11x] pandas scikit-learn
!python -m spacy download en_core_web_sm

/usr/local/lib/python3.12/dist-packages/cupy/_environment.py:447: UserWarning: 
--------------------------------------------------------------------------------

  CuPy may not function correctly because multiple CuPy packages are installed
  in your environment:

    cupy-cuda11x, cupy-cuda12x

  Follow these steps to resolve this issue:

    1. For all packages listed above, run the following command to remove all
       existing CuPy installations:

         $ pip uninstall <package_name>

      If you previously installed CuPy via conda, also run the following:

         $ conda uninstall cupy

    2. Install the appropriate CuPy package.
       Refer to the Installation Guide for detailed instructions.

         https://docs.cupy.dev/en/stable/install.html

--------------------------------------------------------------------------------

  warnings.warn(f'''
/usr/local/lib/python3.12/dist-packages/cupy/_environment.py:447: UserWarning: 
--------------------------------------------

In [2]:
import spacy

try:
    spacy.require_gpu()
    print("✅ Using GPU")
except:
    print("⚠️ GPU not available, using CPU")

/usr/local/lib/python3.12/dist-packages/cupy/_environment.py:447: UserWarning: 
--------------------------------------------------------------------------------

  CuPy may not function correctly because multiple CuPy packages are installed
  in your environment:

    cupy-cuda11x, cupy-cuda12x

  Follow these steps to resolve this issue:

    1. For all packages listed above, run the following command to remove all
       existing CuPy installations:

         $ pip uninstall <package_name>

      If you previously installed CuPy via conda, also run the following:

         $ conda uninstall cupy

    2. Install the appropriate CuPy package.
       Refer to the Installation Guide for detailed instructions.

         https://docs.cupy.dev/en/stable/install.html

--------------------------------------------------------------------------------

  warnings.warn(f'''
/usr/local/lib/python3.12/dist-packages/cupy/_environment.py:447: UserWarning: 
--------------------------------------------

⚠️ GPU not available, using CPU


In [3]:
from google.colab import files
uploaded = files.upload()

Saving Lables.csv to Lables.csv


In [4]:
import pandas as pd

df = pd.read_csv("/content/Lables.csv")
df.head()

,Id,UserId,MessageContent,Time,Sender,MessageId,Type,Amount,Target,Alias,Account
0,b79bbf7c-6e8b-4356-929b-026c8277e390,9c86307b-8354-4575-86e7-89258e57881e,ICICI Bank Acct XX789 debited for Rs 80.00 on ...,2025-12-26 11:44:18+00,AX-ICICIT-S,cc53375a-447d-4b42-a819-7e1c9f6db10b,Debit,80.00,AWFIS,590283159319,XX789
1,ec6680c2-6aa9-4675-8ac2-7cd1f4820a76,9c86307b-8354-4575-86e7-89258e57881e,ICICI Bank Acct XX789 debited for Rs 306.21 on...,2025-12-26 15:24:11+00,AX-ICICIT-S,3cab480f-410f-4482-a685-afb1db248290,Debit,306.21,ZOMATO,989788759216,XX789
2,e9fb8ada-2666-401e-b7be-152c9a9c7fdd,9c86307b-8354-4575-86e7-89258e57881e,A/c *0186 Credited for Rs:5000.00 on 26-12-202...,2025-12-26 15:41:35+00,JD-UNIONB-S,c82bf5ba-4931-4fd3-b711-ff40c1a36c1c,Credit,5000.00,Unknown,Unknown,*0186
3,80da7ab7-8b35-4132-8e33-11a22f124dbf,9c86307b-8354-4575-86e7-89258e57881e,A/c *0186 Debited for Rs:600.00 on 28-12-2025 ...,2025-12-28 15:01:24+00,JD-UNIONB-S,f645a1aa-386c-4872-a25b-569469cabcb7,Debit,600.00,Unknown,Unknown,*0186
4,d290bf24-3959-4cb8-95ab-2b1309b3955c,9c86307b-8354-4575-86e7-89258e57881e,ICICI Bank Acct XX789 debited for Rs 80.00 on ...,2025-12-28 15:21:45+00,AX-ICICIT-S,58f58044-fe67-44f6-9465-23d9938727c4,Debit,80.00,MRS GEDDA RAMA,771066172463,XX789


In [5]:
df["MessageContent"] = df["MessageContent"].astype(str)
df["Sender"] = df["Sender"].astype(str)

# Normalize
df["Target"] = df["Target"].astype(str).str.upper()
df["Account"] = df["Account"].astype(str).str.upper()
df["Type"] = df["Type"].astype(str).str.upper()

In [6]:
import re

def create_training_data(df):
    TRAIN_DATA = []

    for _, row in df.iterrows():
        text = str(row["MessageContent"])
        text_upper = text.upper()
        entities = []

        def add_entity(value, label):
            if pd.isna(value):
                return

            value = str(value).strip()
            if value.lower() == "unknown":
                return

            start = text_upper.find(value.upper())
            if start != -1:
                end = start + len(value)
                entities.append((start, end, label))

        def add_amount(amount):
            if pd.isna(amount):
                return

            amount = str(amount)

            patterns = [
                f"RS {amount}",
                f"RS.{amount}",
                f"RS {amount}.00",
                f"{amount}.00"
            ]

            for p in patterns:
                start = text_upper.find(p.upper())
                if start != -1:
                    entities.append((start, start + len(p), "AMOUNT"))
                    return

        def add_type(text):
            if "DEBITED" in text:
                start = text.find("DEBITED")
                entities.append((start, start + 7, "TYPE"))
            elif "CREDITED" in text:
                start = text.find("CREDITED")
                entities.append((start, start + 8, "TYPE"))

        add_amount(row["Amount"])
        add_entity(row["Account"], "ACCOUNT")
        add_entity(row["Target"], "TARGET")
        add_type(text_upper)

        if entities:
            TRAIN_DATA.append((text, {"entities": entities}))

    return TRAIN_DATA


TRAIN_DATA = create_training_data(df)

print("Training samples:", len(TRAIN_DATA))
print(TRAIN_DATA[:2])

Training samples: 653
[('ICICI Bank Acct XX789 debited for Rs 80.00 on 26-Dec-25; AWFIS credited. UPI:590283159319. Call 18002662 for dispute. SMS BLOCK 789 to 9215676766.', {'entities': [(34, 41, 'AMOUNT'), (16, 21, 'ACCOUNT'), (57, 62, 'TARGET'), (22, 29, 'TYPE')]}), ('ICICI Bank Acct XX789 debited for Rs 306.21 on 26-Dec-25; ZOMATO credited. UPI:989788759216. Call 18002662 for dispute. SMS BLOCK 789 to 9215676766.', {'entities': [(34, 43, 'AMOUNT'), (16, 21, 'ACCOUNT'), (58, 64, 'TARGET'), (22, 29, 'TYPE')]})]


In [7]:
import spacy
from spacy.training.example import Example
import random

nlp = spacy.blank("en")
ner = nlp.add_pipe("ner")

labels = ["AMOUNT", "ACCOUNT", "TARGET", "TYPE"]

for label in labels:
    ner.add_label(label)

optimizer = nlp.begin_training()

for epoch in range(30):
    random.shuffle(TRAIN_DATA)
    losses = {}

    for text, annotations in TRAIN_DATA:
        doc = nlp.make_doc(text)
        example = Example.from_dict(doc, annotations)
        nlp.update([example], losses=losses)

    print(f"Epoch {epoch}: {losses}")

/usr/local/lib/python3.12/dist-packages/spacy/training/iob_utils.py:149: UserWarning: [W030] Some entities could not be aligned in the text "ICICI Bank Acct XX789 debited for Rs 60.00 on 24-F..." with entities "[(34, 41, 'AMOUNT'), (16, 21, 'ACCOUNT'), (57, 72,...". Use `spacy.training.offsets_to_biluo_tags(nlp.make_doc(text), entities)` to check the alignment. Misaligned entities ('-') will be ignored during training.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/spacy/training/iob_utils.py:149: UserWarning: [W030] Some entities could not be aligned in the text "ICICI Bank Acct XX789 debited for Rs 400.00 on 07-..." with entities "[(34, 42, 'AMOUNT'), (16, 21, 'ACCOUNT'), (58, 73,...". Use `spacy.training.offsets_to_biluo_tags(nlp.make_doc(text), entities)` to check the alignment. Misaligned entities ('-') will be ignored during training.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/spacy/training/iob_utils.py:149: UserWarning: [W030] Some entities could not be 

Epoch 0: {'ner': 581.3923919016513}
Epoch 1: {'ner': 2.0203359484763146}
Epoch 2: {'ner': 2.1678198894375855e-06}
Epoch 3: {'ner': 3.6072801881784973e-06}
Epoch 4: {'ner': 1.3410203673868456e-07}
Epoch 5: {'ner': 1.7404213637415204e-06}
Epoch 6: {'ner': 1.8834864905291578e-08}
Epoch 7: {'ner': 4.541788640592274e-08}
Epoch 8: {'ner': 7.728793121548944e-09}
Epoch 9: {'ner': 6.6909868043829464e-09}
Epoch 10: {'ner': 2.9447069498286253e-08}
Epoch 11: {'ner': 9.033450152875622e-09}
Epoch 12: {'ner': 2.2763032067277293e-08}
Epoch 13: {'ner': 1.6985859608521116e-09}
Epoch 14: {'ner': 1.8097152478463145e-09}
Epoch 15: {'ner': 1.318066960699898e-09}
Epoch 16: {'ner': 2.333283373126783e-10}
Epoch 17: {'ner': 7.168785928215266e-11}
Epoch 18: {'ner': 6.65620595038302e-10}
Epoch 19: {'ner': 5.172592643430608e-10}
Epoch 20: {'ner': 3.649500924176779e-11}
Epoch 21: {'ner': 5.881710084927346e-11}
Epoch 22: {'ner': 3.3825867970357165e-12}
Epoch 23: {'ner': 1.5254139091351732e-12}
Epoch 24: {'ner': 1.06

In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

alias_df = df[df["Alias"].notna()]
alias_df = alias_df[alias_df["Alias"] != "Unknown"]

alias_df["input_text"] = (
    alias_df["Sender"] + " " + alias_df["MessageContent"]
)

X = alias_df["input_text"]
y = alias_df["Alias"]

vectorizer = TfidfVectorizer(
    max_features=8000,
    ngram_range=(1,3),
    min_df=2
)

X_vec = vectorizer.fit_transform(X)

model = LogisticRegression(max_iter=1000)
model.fit(X_vec, y)

print("Alias model trained")

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1201: UserWarning: The number of unique classes is greater than 50% of the number of samples. `y` could represent a regression problem, not a classification problem.
  check_classification_targets(y)


Alias model trained


In [9]:
def predict_alias(sender, sms):
    text = sender + " " + sms
    vec = vectorizer.transform([text])

    probs = model.predict_proba(vec)
    confidence = probs.max()

    if confidence < 0.6:
        return "Unknown"

    return model.classes_[probs.argmax()]


def parse_sms(sender, sms):
    doc = nlp(sms)

    result = {
        "Type": None,
        "Amount": None,
        "Target": None,
        "Account": None,
        "Alias": None
    }

    for ent in doc.ents:
        if ent.label_ == "TYPE":
            result["Type"] = ent.text
        elif ent.label_ == "AMOUNT":
            result["Amount"] = ent.text
        elif ent.label_ == "TARGET":
            result["Target"] = ent.text
        elif ent.label_ == "ACCOUNT":
            result["Account"] = ent.text

    result["Alias"] = predict_alias(sender, sms)

    return result

In [10]:
sms = "ICICI Bank Acct XX789 debited for Rs 80.00 on 26-Dec-25; AWFIS credited. UPI:590283159319"
sender = "AX-ICICIT-S"

print(parse_sms(sender, sms))

{'Type': 'debited', 'Amount': 'Rs 80.00', 'Target': 'AWFIS', 'Account': 'XX789', 'Alias': 'Unknown'}


In [12]:
import pickle
import shutil
import os

# Save NER
nlp.to_disk("sms_ner_model")

# Save alias model
pickle.dump(model, open("alias_model.pkl", "wb"))
pickle.dump(vectorizer, open("vectorizer.pkl", "wb"))

# Create a temporary directory for models
models_dir = "./temp_models_to_zip"
os.makedirs(models_dir, exist_ok=True)

# Move model files to the temporary directory
shutil.move("sms_ner_model", models_dir)
shutil.move("alias_model.pkl", models_dir)
shutil.move("vectorizer.pkl", models_dir)

# Zip the temporary directory
shutil.make_archive("models", 'zip', models_dir)

# Clean up the temporary directory
shutil.rmtree(models_dir)


In [13]:
from google.colab import files
files.download("models.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>